In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd

# Task 1: Write your code here:
# Load the dataset
df_path = os.path.join(path, 'Q1_data.csv')

df = pd.read_csv(df_path)

print(f"Dataset shape: {df.shape}")
#we have 1663 samples and 9 feutures

In [ ]:
# Task 2: Write your code here:
# first 5 rows
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
import matplotlib.pyplot as plt

# Task 5: Write your code here:
# Delivery Time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()
#From the plot below we can see that the target a litile  reight skewed

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:

# Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

#since the sample is mor than 1k, I will delete the missing valus for now.
#but if there some time we can try to impute them with mode for catogrical coloumn and see if this will make the model better
#impotentL: the missing target value must drop, we cannt predict somting missing

df = df.dropna(subset=['Weather','Traffic_Level','Time_of_Day','Courier_Experience_yrs','Delivery_Time'])

print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
#Do we have duplicate samples?

def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler, LabelEncoder,OneHotEncoder

# Encode categorical columns - converts text to integers
#LabelEncoder: used for ordnial data, but sometimes we need to use it
#even if the data is nominal, because  if we use OneHotEncoder
#it will increes the number of dimnation and we may fall in problem calld
#curse of dimantionlty, so I will use it only for the Weather

categorical_cols = ['Traffic_Level', 'Time_of_Day', 'Vehicle_Type','Weather']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

#somting go worng with it, If there is time I will come back and try to fix it
#for now I will do LabelEncoder only
#Weather_cat = ['Weather']
#oneHot = OneHotEncoder(sparse_output=False)
#df[Weather_cat]  = oneHot.fit_transform(df[Weather_cat].astype(str))


df.head()

In [ ]:
# Task 5: Write your code here:

features = df.columns.drop("Delivery_Time")  # WE DON'T SCALE THE TARGET
numerical_cols = df[features].select_dtypes(include='number').columns
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
#we are working on linar regration problem so no need for that, If it is a classification we will use it.

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import  mean_absolute_error

# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=40, max_depth=10, random_state=42, n_jobs=-1)

# Storage for linear regression results for each fold
lr_mae = []

n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train.values, y_train.values)

  # Validate
  y_pred = model.predict(X_test.values)

  # Calculate evaluation metrics
  mae = mean_absolute_error(y_test, y_pred)


  # Store results
  lr_mae.append(mae)

print('averaged score across all folds:', np.mean(np.array(mae)))

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
# Retrieve Ridge coefficients and sort by absolute importance

importance = list(zip(X.columns, model.feature_importances_))
sorted_importance = sorted(importance, key=lambda x: abs(x[1]), reverse=True)

# Extract sorted features and their coefficients
features, coefficients = zip(*sorted_importance)

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(features, coefficients, color='darkblue')
plt.xlabel('Coefficient Value')
plt.ylabel('Features')
plt.title('Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=30, edgecolor='black', color='yellow')
plt.title('Delivery time histogram')

plt.show()


In [ ]:
# Task Bonus: Write your code here: